# Spark SQL Assignment

In [2]:
from pyspark.context import SparkContext
from pyspark.sql import SparkSession

In [17]:
import pyspark.sql.types as tp
from pyspark.sql import functions as F

In [3]:
spark=SparkSession.builder.appName("My Application").master("local").getOrCreate()

##### Read CSV files

In [8]:
train_df = spark.read.csv(r"C:\Users\likhi\VSCODE\APACHE-SPARK\data\train.csv", header=True, inferSchema=True)

center_df = spark.read.csv(r"C:\Users\likhi\VSCODE\APACHE-SPARK\data\fulfilment_center_info.csv",
                           header=True,
                           inferSchema=True)

meal_df = spark.read.csv(r"C:\Users\likhi\VSCODE\APACHE-SPARK\data\meal_info.csv",
                         header=True,
                         inferSchema=True)

##### Create temp views

In [9]:
train_df.createOrReplaceTempView("train")

center_df.createOrReplaceTempView("center")

meal_df.createOrReplaceTempView("meal")

##### Q1. What are the distinct number of meal categories and cuisines?

In [22]:
spark.sql("""
SELECT
COUNT(DISTINCT category) AS total_categories ,
COUNT(DISTINCT cuisine) AS total_cuisines
FROM meal;
""").show()

+----------------+--------------+
|total_categories|total_cuisines|
+----------------+--------------+
|              14|             4|
+----------------+--------------+



##### Q2. Which center_id has the highest num_orders?

In [23]:
spark.sql("""
SELECT
center_id,
SUM(num_orders) AS total_orders
FROM train
GROUP BY center_id
ORDER BY total_orders DESC
LIMIT 1;
          
""").show()

+---------+------------+
|center_id|total_orders|
+---------+------------+
|       13|     1742220|
+---------+------------+



##### Q3. What is the top selling cuisine at the center_id that had the highest num_orders?

In [25]:
spark.sql("""
SELECT
m.cuisine,
SUM(t.num_orders) AS total_orders
FROM train t
JOIN meal m
ON t.meal_id = m.meal_id
WHERE t.center_id =
(
SELECT center_id
FROM train
GROUP BY center_id
ORDER BY SUM(num_orders) DESC
LIMIT 1
)
GROUP BY m.cuisine
ORDER BY total_orders DESC
LIMIT 1
          """).show()

+-------+------------+
|cuisine|total_orders|
+-------+------------+
|   Thai|      654724|
+-------+------------+



##### Q4. What is the average op_area per center_type?

In [26]:
spark.sql("""
          SELECT
center_type,
ROUND(AVG(op_area),2) AS avg_op_area
FROM center
GROUP BY center_type
          """).show()

+-----------+-----------+
|center_type|avg_op_area|
+-----------+-----------+
|     TYPE_C|       3.16|
|     TYPE_B|       4.77|
|     TYPE_A|       4.08|
+-----------+-----------+



##### Q5. Which center_type had the highest revenue?

In [27]:
spark.sql("""
SELECT
c.center_type,
SUM(t.checkout_price * t.num_orders) AS revenue
FROM train t
JOIN center c
ON t.center_id = c.center_id
GROUP BY c.center_type
ORDER BY revenue DESC
LIMIT 1;
          """).show()

+-----------+-------------------+
|center_type|            revenue|
+-----------+-------------------+
|     TYPE_A|7.276203201869738E9|
+-----------+-------------------+



##### Q6. Which is the top ordered cuisine in terms of num_orders?

In [29]:
spark.sql("""
SELECT
m.cuisine,
SUM(t.num_orders) AS total_orders
FROM train t
JOIN meal m
ON t.meal_id = m.meal_id
GROUP BY m.cuisine
ORDER BY total_orders DESC
LIMIT 1
          """).show()

+-------+------------+
|cuisine|total_orders|
+-------+------------+
|Italian|    17166334|
+-------+------------+



##### Q7. What are the num_orders per cuisine per week?

In [30]:
spark.sql("""
SELECT
t.week,
m.cuisine,
SUM(t.num_orders) AS total_orders
FROM train t
JOIN meal m
ON t.meal_id = m.meal_id
GROUP BY
t.week,
m.cuisine
ORDER BY
t.week,
m.cuisine
          """).show()

+----+-----------+------------+
|week|    cuisine|total_orders|
+----+-----------+------------+
|   1|Continental|      146020|
|   1|     Indian|      175317|
|   1|    Italian|      228836|
|   1|       Thai|      242088|
|   2|Continental|      133570|
|   2|     Indian|      177109|
|   2|    Italian|      202627|
|   2|       Thai|      273778|
|   3|Continental|       97977|
|   3|     Indian|      150148|
|   3|    Italian|      197299|
|   3|       Thai|      249838|
|   4|Continental|      118819|
|   4|     Indian|      155239|
|   4|    Italian|      192265|
|   4|       Thai|      277206|
|   5|Continental|      116077|
|   5|     Indian|      683532|
|   5|    Italian|      169161|
|   5|       Thai|      229905|
+----+-----------+------------+
only showing top 20 rows


##### Q8. Which center_id gave the highest number of discounts?

In [31]:
spark.sql("""
SELECT
center_id,
COUNT(*) AS discount_count
FROM train
WHERE checkout_price < base_price
GROUP BY center_id
ORDER BY discount_count DESC
LIMIT 1
""").show()

+---------+--------------+
|center_id|discount_count|
+---------+--------------+
|       13|          1509|
+---------+--------------+

